In [ ]:
# Setup
# Import required packages

import os
import sys
import grass.script as gs
import grass.script.setup as gsetup

from concurrent.futures import ThreadPoolExecutor, as_completed
import threading
import psutil
import itertools

import json
import time

# Update paths as required
GISDBASE = r"Grassdata"
PROJECT = "R3NUrb_grass_env"
BASE_MAPSET = "PERMANENT"
COST_RASTER = "EMP_Cost"
POINTS = "EDG_urban"

# Message formatting (purely for intelligible visualisation of progress tracking)
class Colors:
    HEADER = '\033[95m'
    PROCESSING = '\033[94m'
    OK = '\033[92m'
    WARNING = '\033[93m'
    FAIL = '\033[91m'
    END = '\033[0m'
    BOLD = '\033[1m'

# Initialize GRASS session
gsetup.init(GISDBASE, PROJECT, BASE_MAPSET)
print("GRASS GIS session initialized in Jupyter.")


In [ ]:
# Import site shapefile
gs.run_command(
    "v.import",
    input="Data/Input/Vector/EDG_urban/EDG_urban.shp",
    output="EDG_urban",
    overwrite=True,
    quiet=True
)

In [ ]:
# Import Cost surface (NOTE this assumes raster is the same resolution as the project, if not, remove the o flag)
gs.run_command(
   "r.import",
   input="Data/Input/Raster/Cost_surfaces/Land/All_factors/EMP_ALLFAC_AUG_93m_SEA45.tif",
   output="EMP_Cost",
   flags="o",
   memory= 50000,
   overwrite=True,
   quiet=True
  )

# Set the computational region to the cost raster
gs.run_command("g.region", raster="EMP_Cost", quiet=True)

In [ ]:
# User-configurable parameters for memory use
NUM_PARALLEL_PROCESSES = 1   # Number of r.cost tasks to run in parallel
RCOST_MEMORY_MB = 45000       # Memory (MB) for each r.cost task

# Check total requested memory
total_requested_mem_gb = (NUM_PARALLEL_PROCESSES * RCOST_MEMORY_MB) / 1024
available_mem_gb = psutil.virtual_memory().available / (1024 ** 3)
if total_requested_mem_gb > available_mem_gb:
    print(f"{Colors.WARNING}⚠ Warning: Total requested memory ({total_requested_mem_gb:.1f} GB) "
          f"exceeds available RAM ({available_mem_gb:.1f} GB). Reduce parallel tasks or per-task memory.{Colors.END}")

In [ ]:
# Cumulative Cost Surface Generation (parallel r.cost)
# Thread lock for safe printing
print_lock = threading.Lock()

# Define functions
# Extract single points sequentially
def extract_point(point_cat, name):
    single_point = f"point_{point_cat}"
    gs.run_command("v.extract",
                   input=f"{POINTS}@{BASE_MAPSET}",
                   output=single_point,
                   where=f"cat={point_cat}",
                   quiet=True,
                   overwrite=True)
    return (point_cat, name, single_point)

# Run r.cost for a single extracted point
def generate_cost_surface(args, total_points):
    point_cat, name, single_point = args
    output_name = str(name).replace(" ", "_").replace("(", "").replace(")", "")
    with print_lock:
        print(f"{Colors.PROCESSING}➤ Starting r.cost for {point_cat}: {output_name}{Colors.END}", flush=True)
    try:
        gs.run_command("r.cost",
                       input=COST_RASTER,
                       output=f"Cum_cost_{output_name}",
                       start_points=single_point,
                       flags="k",
                       quiet=True,
                       memory=RCOST_MEMORY_MB,
                       overwrite=False)
        with print_lock:
            print(f"{Colors.OK}✅ Completed r.cost for {point_cat}: {output_name}{Colors.END}", flush=True)
    except Exception as e:
        with print_lock:
            print(f"{Colors.FAIL}❌ Failed r.cost for {point_cat}: {output_name} ({e}){Colors.END}", flush=True)

# Main execution of Cost Corridor functions
if __name__ == "__main__":
    # Get the point IDs (cat) and Safe_name
    query = gs.read_command("v.db.select",
                            map=f"{POINTS}@{BASE_MAPSET}",
                            columns="cat,Safe_name").splitlines()[1:]  # Skip header

    # Build list of (cat, name)
    points = [(int(row.split("|")[0]), row.split("|")[1]) for row in query if row.split("|")[0].isdigit()]

    print(f"{Colors.HEADER}\n{'='*60}")
    print(f"Extracting {len(points)} points sequentially...")
    print(f"{'='*60}{Colors.END}")

    # Step 1: Sequentially extract points
    extracted_points = []
    for cat, name in points:
        with print_lock:
            print(f"{Colors.PROCESSING}➤ Extracting point {cat}: {name}{Colors.END}", flush=True)
        extracted_points.append(extract_point(cat, name))
    print(f"{Colors.OK}✅ All points extracted.{Colors.END}")

    # Step 2: Parallel r.cost computation
    print(f"{Colors.HEADER}\n{'='*60}")
    print(f"Generating cost surfaces in parallel ({NUM_PARALLEL_PROCESSES} tasks)...")
    print(f"{'='*60}{Colors.END}")

    with ThreadPoolExecutor(max_workers=NUM_PARALLEL_PROCESSES) as executor:
        futures = [executor.submit(generate_cost_surface, p, len(extracted_points)) for p in extracted_points]
        for future in as_completed(futures):
            pass 

    # Step 3: Remove all temporary point vectors
    print(f"{Colors.HEADER}\n{'='*60}")
    print(f"Cleaning up temporary point vectors...")
    print(f"{'='*60}{Colors.END}")

    for _, _, single_point in extracted_points:
        gs.run_command("g.remove", type="vector", name=single_point, flags="f", quiet=True)

    print(f"{Colors.OK}✅ All temporary points removed. Process complete.{Colors.END}")


In [ ]:
# Cost Corridor Generation and Normalisation
# Define functions
def get_rasters(pattern="Cum_cost_"):
    """Return all rasters that start with pattern"""
    rasters = gs.list_grouped("rast")[gs.gisenv()["MAPSET"]]
    return [r for r in rasters if r.startswith(pattern)]

def process_pair(r1, r2):
    """Compute average and normalised rasters for a given pair"""
    short1 = r1.replace("Cum_cost_", "")
    short2 = r2.replace("Cum_cost_", "")

    avg_name = f"CORR_{short1}_TO_{short2}"
    norm_name = f"NORM_CORR_{short1}_TO_{short2}"

    print(f"{Colors.PROCESSING} ⚙️ Processing pair: {avg_name}{Colors.END}", flush=True)

    # 1 Compute average raster
    expr_avg = f"{avg_name} = ({r1} + {r2}) / 2.0"
    gs.run_command("r.mapcalc", expression=expr_avg, overwrite=True)
    print(f"{Colors.OK}✅ Corridor Generated: {avg_name}.{Colors.END}", flush=True)

    # 2 Get stats for normalisation
    stats = gs.parse_command("r.univar", map=avg_name, flags="g")
    min_val = float(stats["min"])
    max_val = float(stats["max"])

    print(f"{Colors.PROCESSING} ⚙️ Normalising corridor: {avg_name}, with original min: {min_val} and max: {max_val}{Colors.END}", flush=True)

    # 3 Normalise to 1–1000
    expr_norm = (
        f"{norm_name} = (({avg_name} - {min_val}) / ({max_val} - {min_val})) * 999.0 + 1.0"
    )
    gs.run_command("r.mapcalc", expression=expr_norm, overwrite=True)

    print(f"{Colors.OK}✅ Normalisation successful: {norm_name} created.{Colors.END}", flush=True)
    return norm_name

def main():
    # ---------------------------------------------------------
    # 1 Find all Cum_cost_* rasters
    # ---------------------------------------------------------
    rasters = get_rasters("Cum_cost_")
    if len(rasters) < 2:
        gs.fatal("Need at least two rasters named 'Cum_cost_*'")

    print(f"{Colors.PROCESSING} 📂 Found {len(rasters)} rasters:{Colors.END}", flush=True)
    for r in rasters:
        print(f"   - {r}")

    # ---------------------------------------------------------
    # 2 Create all unique pairs (A,B) without repetition
    # ---------------------------------------------------------
    pairs = list(itertools.combinations(rasters, 2))
    print(f"{Colors.PROCESSING}➤ {len(pairs)} unique pairs to process{Colors.END}", flush=True)

    # ---------------------------------------------------------
    # 3 Parallel execution
    # ---------------------------------------------------------
    n_jobs = min(80, len(pairs))  # Use up to 80 threads
    print(f"{Colors.PROCESSING}➤ Running with {n_jobs} parallel workers{Colors.END}", flush=True)

    with concurrent.futures.ThreadPoolExecutor(max_workers=n_jobs) as executor:
        futures = [executor.submit(process_pair, r1, r2) for r1, r2 in pairs]
        results = [f.result() for f in concurrent.futures.as_completed(futures)]

    print(f"{Colors.OK}✅ All pairs processed successfully{Colors.END}", flush=True)
    print(f"{Colors.OK}✅ Normalised outputs:{Colors.END}", flush=True)
    for r in results:
        print(f"   - {r}")

if __name__ == "__main__":
    main()
     

In [ ]:
#!/usr/bin/env python3
# Batch merging of cost corridor (with many large rasters to merge, a single r.series merge is impractical. For smaller case studies, this step can be skipped/simplified)
# Parameters
TEMP_PREFIX = "TEMP_PAIR_ALLFAC_AUG"
FINAL_NAME = "MINNORM_CORRIDORS_ALLFAC_AUG"
STATE_FILE = "pairwise_state_ALLFAC_AUG.json"
MAX_WORKERS = 100          # Number of concurrent merges
MEMORY_MB = 4500           # r.series memory
RETRIES = 1                # Retries per merge

# Define helper functions
def log(msg):
    print(msg, flush=True)

def raster_exists(name):
    """Check if a raster exists in current mapset."""
    try:
        return name in gs.list_grouped("rast")[gs.gisenv()["MAPSET"]]
    except:
        return False

def safe_delete(rasters):
    """Safely delete temporary rasters if they exist."""
    for r in rasters:
        if raster_exists(r):
            try:
                gs.run_command("g.remove", type="rast", name=r, flags="f", quiet=True)
            except Exception as e:
                log(f"{Colors.WARNING}⚠️ Could not remove {r}: {e}{Colors.END}")

# Define merge functions
def merge_pair(r1, r2, out_name):
    """Run r.series for a pair of rasters with retry & logging."""
    for attempt in range(RETRIES + 1):
        try:
            log(f"{Colors.PROCESSING}➤ Merging '{r1}' + '{r2}' → {out_name}{Colors.END}")
            if not raster_exists(r1) or not raster_exists(r2):
                raise RuntimeError(f"Missing input raster(s): {r1 if not raster_exists(r1) else ''} {r2 if not raster_exists(r2) else ''}")
            gs.run_command(
                "r.series",
                input=f"{r1},{r2}",
                output=out_name,
                method="minimum",
                memory=MEMORY_MB,
                overwrite=True,
                quiet=True,
            )
            log(f"{Colors.OK}✅ Merge complete → {out_name}{Colors.END}")
            return (out_name, None)
        except Exception as e:
            log(f"{Colors.FAIL}❌ Error merging {r1}+{r2} → {out_name}: {e}{Colors.END}")
            if attempt < RETRIES:
                log(f"↻ Retry {attempt+1}/{RETRIES} for {r1}+{r2} → {out_name}", flush=True)
                time.sleep(5)
            else:
                return (out_name, (r1, r2, str(e)))


def merge_triple(r1, r2, r3, out_name):
    """Run r.series for three rasters (triple merge)."""
    for attempt in range(RETRIES + 1):
        try:
            log(f"{Colors.PROCESSING}➤ Triple merge '{r1}' + '{r2}' + '{r3}' → {out_name}{Colors.END}")
            if not all(raster_exists(r) for r in [r1, r2, r3]):
                missing = [r for r in [r1, r2, r3] if not raster_exists(r)]
                raise RuntimeError(f"Missing input raster(s): {', '.join(missing)}")
            gs.run_command(
                "r.series",
                input=f"{r1},{r2},{r3}",
                output=out_name,
                method="minimum",
                memory=MEMORY_MB,
                overwrite=True,
                quiet=True,
            )
            log(f"{Colors.OK}✅ Triple merge complete → {out_name}{Colors.END}")
            return (out_name, None)
        except Exception as e:
            log(f"{Colors.FAIL}❌ Error triple merging {r1}+{r2}+{r3} → {out_name}: {e}{Colors.END}")
            if attempt < RETRIES:
                log(f"↻ Retry {attempt+1}/{RETRIES} for triple {r1}+{r2}+{r3} → {out_name}", flush=True)
                time.sleep(5)
            else:
                return (out_name, (r1, r2, r3, str(e)))

# Merge Reduction Loop
def reduce_rasters(rasters):
    """Iteratively merge rasters by min cell stats until one remains."""
    round_idx = 0
    current = list(rasters)

    # Resume if state file exists
    if os.path.exists(STATE_FILE):
        with open(STATE_FILE) as f:
            saved = json.load(f)
        if saved.get("remaining"):
            current = saved["remaining"]
            round_idx = saved["round_idx"]
            log(f"{Colors.WARNING}⚠️ Resuming from Round {round_idx} with {len(current)} rasters...{Colors.END}")

    # Loop until one remains
    while len(current) > 1:
        round_idx += 1
        log(f"\n{Colors.HEADER}{Colors.BOLD}=== ROUND {round_idx} | items={len(current)} ==={Colors.END}")

        # Pre-round validation
        valid_current = [r for r in current if raster_exists(r)]
        if len(valid_current) < len(current):
            log(f"{Colors.WARNING}⚠️ Dropped {len(current)-len(valid_current)} missing rasters before round {round_idx}{Colors.END}")
        current = valid_current

        # Create corridor merge pairs
        pairs = []
        next_round = []
        for i in range(0, len(current) - 1, 2):
            r1, r2 = current[i], current[i+1]
            out_name = f"{TEMP_PREFIX}_R{round_idx:02d}_P{i//2:04d}"
            pairs.append((r1, r2, out_name))

        # Handle leftover unpaired raster as a triple merge
        if len(current) % 2 == 1:
            leftover = current[-1]
            if pairs:
                r1, r2, out_name = pairs.pop()
                out_name_triple = f"{TEMP_PREFIX}_R{round_idx:02d}_P_TRIPLE"
                log(f"{Colors.WARNING}⚠️ Combining leftover raster '{leftover}' with last pair ({r1}, {r2}) → triple merge{Colors.END}")
                pairs.append((r1, r2, leftover, out_name_triple))
            else:
                log(f"{Colors.WARNING}⚠️ Only one raster remains, carrying over: {leftover}{Colors.END}")
                next_round.append(leftover)

        log(f"🔁 Round {round_idx}: {len(pairs)} pairs, concurrency={MAX_WORKERS}")

        # Parallel merge execution
        results = []
        failed = []
        with concurrent.futures.ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
            futures = {}
            for p in pairs:
                # Skip if output already exists
                if len(p) == 3:
                    r1, r2, out = p
                    if raster_exists(out):
                        log(f"{Colors.WARNING}⏭️ Skipping existing output raster {out}{Colors.END}")
                        next_round.append(out)
                        continue
                    futures[executor.submit(merge_pair, r1, r2, out)] = (r1, r2, out)
                elif len(p) == 4:
                    r1, r2, r3, out = p
                    if raster_exists(out):
                        log(f"{Colors.WARNING}⏭️ Skipping existing output raster {out}{Colors.END}")
                        next_round.append(out)
                        continue
                    futures[executor.submit(merge_triple, r1, r2, r3, out)] = (r1, r2, out)

            for idx, future in enumerate(concurrent.futures.as_completed(futures)):
                r1, r2, out = futures[future]
                out_name, err = future.result()
                if err:
                    failed.append((r1, r2, out, err))
                else:
                    next_round.append(out_name)
                log(f"📈 Round {round_idx} progress: {idx+1}/{len(pairs)} ({(idx+1)/len(pairs)*100:.1f}%)")

        # Handle failed merges safely
        for (r1, r2, out, err) in failed:
            log(f"{Colors.WARNING}⚠️ Handling failed pair: {r1} + {r2}{Colors.END}")
            r1_exists = raster_exists(r1)
            r2_exists = raster_exists(r2)
            if not r1_exists and not r2_exists:
                log(f"{Colors.FAIL}❌ Both inputs missing; dropping failed pair entirely{Colors.END}")
                continue
            if r1_exists and r2_exists:
                next_round.extend([r1, r2])
            elif r1_exists:
                log(f"{Colors.WARNING}⚠️ Only {r1} exists; promoting it forward alone{Colors.END}")
                next_round.append(r1)
            elif r2_exists:
                log(f"{Colors.WARNING}⚠️ Only {r2} exists; promoting it forward alone{Colors.END}")
                next_round.append(r2)

        # Save state
        state = {"round_idx": round_idx, "remaining": next_round}
        with open(STATE_FILE, "w") as f:
            json.dump(state, f, indent=2)

        # Safe cleanup: only if >2 rasters remain
        if len(next_round) > 2:
            old_temp = [r for r in current if r.startswith(TEMP_PREFIX)]
            if old_temp:
                log(f"{Colors.WARNING}🧹 Removing {len(old_temp)} temporary rasters (safe cleanup)...{Colors.END}")
                safe_delete(old_temp)
        else:
            log(f"{Colors.WARNING}🛑 Skipping cleanup — fewer than 3 rasters remain (final rounds){Colors.END}")

        # Prepare for next round
        current = next_round

    # Final processing of merged corridors
    if len(current) == 1:
        final = current[0]
        log(f"{Colors.OK}{Colors.BOLD}🏁 Reduction complete: {final}{Colors.END}")
        if final != FINAL_NAME:
            try:
                gs.run_command("g.rename", raster=f"{final},{FINAL_NAME}")
            except Exception:
                gs.run_command("r.mapcalc", expression=f"{FINAL_NAME} = {final}", overwrite=True)
        log(f"{Colors.OK}✅ Final raster: {FINAL_NAME}{Colors.END}")
        return FINAL_NAME
    else:
        log(f"{Colors.FAIL}❌ Reduction ended prematurely; {len(current)} rasters remain{Colors.END}")
        return None

if __name__ == "__main__":
    log(f"{Colors.HEADER}{Colors.BOLD}🌍 Starting safe pairwise minimum merge...{Colors.END}")
    rasters = [r for r in gs.list_grouped("rast")[gs.gisenv()["MAPSET"]] if r.startswith("NORM_CORR_")]
    if not rasters:
        gs.fatal("No NORM_CORR_* rasters found.")
    log(f"{Colors.PROCESSING}Found {len(rasters)} rasters to merge.{Colors.END}")

    reduce_rasters(rasters)
